In [ ]:
import pandas as pd

In [ ]:
from datasets import load_dataset
ds = load_dataset("AgentPublic/cnil")

# Etape 1 : Exploration de la data

In [ ]:
print(ds)
df = ds["train"].to_pandas()
print(df.head(5))
print(df["text"][0])

In [ ]:
# Combien de décisions
print("df[doc_id].nunique() : ", df["doc_id"].nunique())

In [ ]:
# Nature des sanctions 
print("sanctions :", df["nature"].value_counts())
print("sanctions :", df["nature_delib"].value_counts())

In [ ]:
# Types 
df.dtypes

In [ ]:

# Convert date to datetime 
df["date"] = pd.to_datetime(df["date"])

# Remove leading/trailing whitespace in the number and full_title columns
df["number"] = df["number"].str.strip()
df["full_title"] = df["full_title"].str.strip()

# We will only consider decisions with a sanction or a mise en demeure (nature_delib) for the graph
df_sanctions = df[(df["nature_delib"].isin(["Mise en demeure","Sanction"])) & (df["date"] >= "2018-05-25")].copy()
print("Number of sanctions and mise en demeure:", df_sanctions["doc_id"].nunique())

# Min and max date
print("Minimal date :", df_sanctions["date"].min())
print("Maximal date :", df_sanctions["date"].max())

# Adding the column year to the dataframe
df_sanctions["year"] = df_sanctions["date"].dt.year

# Number of sanctions and mise en demeure per year
df_sanctions_year = df_sanctions.groupby(["year","nature_delib"])["doc_id"].nunique().reset_index(name="nb_decisions")
print("Number of decisions per year and type :")
print(df_sanctions_year)

# Rules (recommendations and standards) for the vector search
df_rules_all = df[df["nature_delib"].isin(["Recommandation/Lignes directrices","Référentiel/Règlement type/Norme"]) & (df["date"] >= "2018-05-25")].copy()
print("Number of rules:", df_rules_all["doc_id"].nunique())

df_rules_all["year"] = df_rules_all["date"].dt.year



In [ ]:
# Rules to keep for PME 

rules_to_keep = [
    "2020-091", "2020-092", "2025-131", "2026-042", "2026-114",
    "2019-160", "2022-126", "2026-031", "2026-054", "2023-064", "2019-001", "2025-028",
    "2022-100", "2025-019", "2021-122", "2023-050", "2026-022",
    "2018-326", "2018-327", "2019-011", "2019-118",
    "2021-131", "2021-130", "2018-303", "2021-057",
    "2025-024", "2024-011", "2025-010", "2025-041", "2025-047", "2024-041",
    "2026-063", "2021-043",
    "2020-081", "2022-067", "2020-076",
    "2021-070", "2026-045",
]

df_rules = df_rules_all[df_rules_all["number"].isin(rules_to_keep)].copy()

print("Number of rules after filtering for PME:", df_rules["doc_id"].nunique()) # 2 are missing

# Check if all rules to keep are present in the dataframe
# Note: some numbers had trailing spaces (e.g. '2026-022 '), fixed with .str.strip()
keep = set(rules_to_keep)
end = set(df_rules["number"].unique())
print("Rules to keep but not found in the dataframe:", keep - end)

found = df[df["full_title"].str.contains("2026-022")]
print(found["number"].unique())

found = df[df["full_title"].str.contains("2026-063")]
print(found["number"].unique())



In [ ]:
# Empty value
print("Empty values in sanctions dataframe:")
print(df_sanctions.isna().sum())

print("Empty values in rules dataframe:")
print(df_rules.isna().sum())

# Status , Nature ? 
print(df_sanctions["nature"].value_counts())
print(df_sanctions["status"].value_counts())

# Drop columns 
cols_to_drop = ["chunk_id", "chunk_xxh64", "nature", "status", "chunk_text", "embeddings_bge-m3"]

df_sanctions = df_sanctions.drop(columns=cols_to_drop)
df_rules = df_rules.drop(columns=cols_to_drop)

print("Columns in sanctions dataframe:", df_sanctions.columns)
print("Columns in rules dataframe:", df_rules.columns)

# Etape 1 (suite) : regroupement des chunks par décision

Une ligne par décision (sanctions et mises en demeure) pour l'extraction par le LLM


In [ ]:
# Sort chunks by position, then merge them into one row per decision
df_sanctions_sorted = df_sanctions.sort_values(by=["doc_id", "chunk_index"])
df_sanctions_grouped = df_sanctions_sorted.groupby(["doc_id"]).agg({"nature_delib": "first", "title":"first","full_title":"first","number":"first","date": "first", "text": " ".join,"year":"first"}).reset_index()

print("The size of df_sanctions_grouped is :", len(df_sanctions_grouped))

print("Five first lines of df_sanctions_grouped are :")
print(df_sanctions_grouped.head(5))

print("Last words of the first decision text :") # The fine amount is present; company names are anonymized
print(df_sanctions_grouped["text"].iloc[0][-1000:])



In [ ]:
# Size of the text
len_text = df_sanctions_grouped["text"].str.len()
print("Size of the text :")
print(len_text)

print("Max size of the text :", len_text.max())
print("Min size of the text :", len_text.min())
print("Mean size of the text :", len_text.mean())

print("Number of decisions with text size < 1000 :", (len_text < 1000).sum())
print("Decisions with text size < 1000 :")
print(df_sanctions_grouped[df_sanctions_grouped["text"].str.len() < 1000])

short = df_sanctions_grouped[len_text < 1000]
print(short["text"].iloc[0])

In [ ]:
# Remove closure letters (no useful content)
df_sanctions_grouped = df_sanctions_grouped[~df_sanctions_grouped["text"].str.contains("clôture du dossier")]

print(len(df_sanctions_grouped), "decisions after removing closure letters")

In [ ]:
# Anonymized companies ? 

df_2025 = df_sanctions_grouped[df_sanctions_grouped["year"] == 2025]
text_2025 = df_2025["text"].iloc[0]   
print("Text of the 2025 decision :")
print(text_2025[-1000:])  # Print the last 1000 characters of the text

# Company names are visible in recent decisions, but anonymized ("X") after 2 years

In [ ]:
# df_rules 

print(df_rules["full_title"].nunique())

df_rules_sorted = df_rules.sort_values(by=["doc_id", "chunk_index"])
df_rules_grouped = df_rules_sorted.groupby(["doc_id"]).agg({"nature_delib": "first", "title":"first","full_title":"first","number":"first","date": "first", "text": " ".join,"year":"first"}).reset_index()

# Affiche tous les titres de règles 

for title in df_rules_grouped["full_title"]:
    print(title)

# empty title (blank lines in the list came from titles ending with "\n", now stripped)
empty_title = df_rules_all[df_rules_all["full_title"] == ""]
print("Empty title in rules dataframe :")
print(empty_title)


In [ ]:
# Chunk text size in rules dataframe

len_text_rules = df_rules_grouped["text"].str.len()
print("Size of the text in rules dataframe :")
print(len_text_rules)

print("Max size of the text in rules dataframe :", len_text_rules.max())
print("Min size of the text in rules dataframe :", len_text_rules.min())
print("Mean size of the text in rules dataframe :", len_text_rules.mean())




In [ ]:
# size 2000 little : strange 
print("Number of rules with text size < 5000 :", (len_text_rules < 5000).sum())
rules_5000 = df_rules_grouped[df_rules_grouped["text"].str.len() < 5000]
print("Rules with text size < 5000 :")
print(rules_5000["text"].iloc[0])

for number, text in zip(rules_5000["number"], rules_5000["text"]):
    print(number, "-", len(text), "characters")
    print(text[-300:])
    print("-" * 50)

# useless chunk (no useful content)

In [ ]:
# Rules without content: annex missing from the dataset, or simple modifications
rules_without_content = [
    "2019-011", "2020-076", "2024-011", "2024-041", "2025-010",
    "2025-019", "2025-024", "2025-028", "2025-041", "2025-047",
    "2026-022", "2026-031", "2026-042", "2026-045", "2026-054",
    "2026-063", "2026-114",
]

df_rules = df_rules[~df_rules["number"].isin(rules_without_content)].copy()
print("Number of rules with content:", df_rules["doc_id"].nunique())


In [ ]:
# Chunk size in rules (chunks, not full documents) /  checking if some chunks are very short (e.g. just a signature or the end of a document), we want clean chunks for the vector search
len_chunks_rules = df_rules["text"].str.len()
print("Rules chunks - min:", len_chunks_rules.min())
print("Rules chunks - max:", len_chunks_rules.max())
print("Rules chunks - mean:", round(len_chunks_rules.mean()))

# Very short chunks: often just a signature or the end of a document
short_chunks = df_rules[len_chunks_rules < 100]
print("Number of chunks < 100 characters:", len(short_chunks))

for number, text in zip(short_chunks["number"], short_chunks["text"]):
    print(number, "-", repr(text))
    print("-" * 50)

# Same for sanctions dataframe
len_chunks_sanctions = df_sanctions["text"].str.len()
print("Sanctions chunks - min:", len_chunks_sanctions.min())
print("Sanctions chunks - max:", len_chunks_sanctions.max())
print("Sanctions chunks - mean:", round(len_chunks_sanctions.mean()))

short_chunks_sanctions = df_sanctions[len_chunks_sanctions < 100]
print("Number of sanctions chunks < 100 characters:", len(short_chunks_sanctions))

for number, text in zip(short_chunks_sanctions["number"], short_chunks_sanctions["text"]):
    print(number, "-", repr(text))
    print("-" * 50)

# Remove very short chunks (sentence fragments with no meaning on their own)
df_sanctions = df_sanctions[df_sanctions["text"].str.len() >= 100].copy()

# Remove closure letters: keep only decisions still in df_sanctions_grouped
df_sanctions = df_sanctions[df_sanctions["doc_id"].isin(df_sanctions_grouped["doc_id"])].copy()

print("Number of chunks:", len(df_sanctions))
print("Number of decisions:", df_sanctions["doc_id"].nunique())

In [ ]:
# Save the cleaned data for the next steps
df_sanctions_grouped.to_parquet("../data/sanctions_decisions.parquet", index=False)
df_sanctions.to_parquet("../data/sanctions_chunks.parquet", index=False)
df_rules.to_parquet("../data/rules_chunks.parquet", index=False)

In [ ]:
# Check: reload the files
check = pd.read_parquet("../data/sanctions_decisions.parquet")
print(len(check), "decisions")

check_chunks = pd.read_parquet("../data/sanctions_chunks.parquet")
print(len(check_chunks), "sanction chunks")

check_rules = pd.read_parquet("../data/rules_chunks.parquet")
print(check_rules["doc_id"].nunique(), "rules")
